# FASE 1 — Carga, Auditoría Inicial y Limpieza del Dataset
## TFG: Predicción de Tráfico Urbano (M30 + URB)

**Targets:** `intensidad_trafico`, `ocupacion`, `carga`  
**Dataset:** `data/Trafico_con_accidentes.csv` · ~37.5M filas · todos los sensores

---
### Índice
1. Imports y configuración
2. Carga exploratoria (muestra)
3. Auditoría inicial
4. Problemas detectados
5. Conversión de tipos
6. Limpieza y tratamiento de nulos / valores erróneos
7. Carga completa con dtypes optimizados
8. Resumen de riesgos de data leakage

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_PATH = "data/Trafico_con_accidentes.csv"
SAMPLE_ROWS = 500_000  # muestra para auditoría inicial (el full son ~37.5M filas)

## 2. Carga exploratoria (muestra)

Se usa `nrows` para la auditoría inicial. El dataset completo (23 GB) se cargará en la sección 7 con dtypes optimizados.

In [ ]:
df = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS, encoding="utf-8", low_memory=False)

print(f"Shape muestra: {df.shape}")
print(f"\n{'='*60}")
print("PRIMERAS 3 FILAS")
print("="*60)
df.head(3)

## 3. Auditoría inicial

In [ ]:
# --- 3.1 Tipos inferidos por pandas ---
print("TIPOS INFERIDOS POR PANDAS")
print("="*60)
print(df.dtypes)
print(f"\nTotal columnas: {df.shape[1]}")

In [ ]:
# --- 3.2 Info completa ---
df.info(memory_usage="deep")

In [ ]:
# --- 3.3 Estadísticas descriptivas de columnas numéricas ---
print("DESCRIBE — columnas numéricas")
print("="*60)
df.describe()

In [ ]:
# --- 3.4 Distribución de columnas categóricas ---
cat_cols = ["tipo_elem", "Dia_semana",
            "laborable / festivo / domingo festivo",
            "Tipo de Festivo", "intensidad_lluvia"]

for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).to_string())

In [ ]:
# --- 3.5 Nulos por columna ---
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "% nulos": pct_nulos
}).query("nulos > 0").sort_values("% nulos", ascending=False)

print("COLUMNAS CON NULOS")
print("="*60)
print(resumen_nulos.to_string() if not resumen_nulos.empty else "Sin nulos en la muestra")

## 4. Problemas detectados

### 4.1 Encoding (Mojibake)
### 4.2 Valores centinela (-1)
### 4.3 Inconsistencias lluvia
### 4.4 Outliers en targets
### 4.5 Columnas redundantes / de baja utilidad

In [ ]:
# --- 4.1 Mojibake: columnas con chars corruptos (Ã©, Ã³, etc.) ---
# El CSV está en UTF-8 pero el texto fue generado con doble codificación latin-1 → UTF-8.
# Síntoma: "Ã©" en lugar de "é", "Ã³" en lugar de "ó".

def fix_mojibake(s):
    """Recupera texto dañado por doble-codificación UTF-8/latin-1."""
    try:
        return s.encode("latin-1").decode("utf-8")
    except Exception:
        return s

str_cols_to_check = ["nombre", "Dia_semana", "Festividad"]
for col in str_cols_to_check:
    sample_val = df[col].dropna().iloc[0]
    fixed = fix_mojibake(sample_val)
    tiene_garbled = "Ã" in str(sample_val)
    print(f"{col:25s}  garbled={tiene_garbled}  antes='{sample_val[:35]}'  →  '{fixed[:35]}'")

# Verificación rápida: cuántos nombres tienen chars corruptos
n_garbled = df["nombre"].str.contains("Ã", na=False).sum()
print(f"\nFilas con 'nombre' corrupto: {n_garbled:,} ({n_garbled/len(df)*100:.1f}%)")

In [ ]:
# --- 4.2 Valores centinela -1 ---
# tiempo_desde_accidente usa -1 para indicar "sin accidente reciente".
# Es un valor semántico válido pero debe distinguirse de un nulo real.

print("VALORES CENTINELA -1 POR COLUMNA")
print("="*60)
numeric_cols = df.select_dtypes(include="number").columns
for col in numeric_cols:
    n_minus1 = (df[col] == -1).sum()
    if n_minus1 > 0:
        pct = n_minus1 / len(df) * 100
        print(f"  {col:35s}: {n_minus1:>8,} filas ({pct:.1f}%)")

print("\n→ DECISIÓN: tiempo_desde_accidente = -1 significa 'sin accidente' (no es nulo).")
print("  Se mantendrá como valor numérico válido. No se reemplaza por NaN.")

In [ ]:
# --- 4.3 Inconsistencia crítica: llueve vs intensidad_lluvia ---
# llueve debería ser 1 cuando intensidad_lluvia != 'no_lluvia'.
# Se detectó que llueve=0 incluso cuando intensidad_lluvia='leve'.

n_inconsistente = ((df["llueve"] == 0) & (df["intensidad_lluvia"] != "no_lluvia")).sum()
total_lluvia = (df["intensidad_lluvia"] != "no_lluvia").sum()

print(f"Filas donde llueve=0 pero intensidad_lluvia≠'no_lluvia': {n_inconsistente:,}")
print(f"Total filas con lluvia según intensidad_lluvia: {total_lluvia:,}")
print(f"Tasa de inconsistencia: {n_inconsistente/max(total_lluvia,1)*100:.1f}%")
print()

# Tabla cruzada para ver la relación
cross = pd.crosstab(df["llueve"], df["intensidad_lluvia"], margins=True)
print("Crosstab llueve × intensidad_lluvia (muestra):")
print(cross)
print()
print("→ DECISIÓN: 'llueve' es la columna corrupta.")
print("  Se reconstruirá como: llueve = (intensidad_lluvia != 'no_lluvia').astype(int)")

In [ ]:
# --- 4.4 Outliers en variables target y vmed ---
targets_y_vmed = ["intensidad_trafico", "ocupacion", "carga", "vmed"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, targets_y_vmed):
    data = df[col].dropna()
    p99 = data.quantile(0.99)
    ax.hist(data[data <= p99], bins=60, edgecolor="none", alpha=0.8, color="steelblue")
    ax.set_title(f"{col}\n(hasta p99={p99:.0f})")
    ax.set_xlabel("valor")
    ax.set_ylabel("frecuencia")

plt.suptitle("Distribución de targets + vmed (hasta percentil 99)", fontsize=13)
plt.tight_layout()
plt.show()

# Percentiles clave
print("\nPERCENTILES TARGETS + VMED")
print("="*60)
for col in targets_y_vmed:
    q = df[col].quantile([0, 0.25, 0.50, 0.75, 0.95, 0.99, 1.0])
    zeros_pct = (df[col] == 0).sum() / len(df) * 100
    print(f"\n{col}  (zeros: {zeros_pct:.1f}%)")
    print(q.to_string())

In [ ]:
# --- 4.5 vmed por tipo_elem ---
# vmed es ~0 en sensores URB (no miden velocidad). Solo M30 tiene vmed válida.
print("VMED POR TIPO_ELEM")
print("="*60)
print(df.groupby("tipo_elem")["vmed"].agg(["count", "mean", "median", lambda x: (x==0).mean()]).rename(
    columns={"<lambda_0>": "% zeros"}
).round(3))
print()
print("→ CONCLUSIÓN: vmed de sensores URB es siempre 0 (no se mide).")
print("  No es útil como feature para modelos que incluyan filas URB.")
print("  Opciones: (a) modelar M30 y URB por separado,")
print("            (b) usar vmed solo para sensores M30 (con NaN para URB).")

# Nulos en vmed
print(f"\nNulos en vmed: {df['vmed'].isnull().sum():,} ({df['vmed'].isnull().mean()*100:.2f}%)")

In [ ]:
# --- 4.6 Accidentes: desbalance extremo ---
print("DESBALANCE EN VARIABLES DE ACCIDENTE")
print("="*60)
acc_cols = ["hay_accidente", "heridos_leves", "heridos_graves", "victimas_mortales"]
for col in acc_cols:
    n_pos = (df[col] > 0).sum()
    print(f"  {col:30s}: {n_pos:>8,} filas con valor>0  ({n_pos/len(df)*100:.3f}%)")

print()
print("→ hay_accidente y variables de gravedad están extremadamente desbalanceadas.")
print("  Si el modelo necesita estas features, considerar sub-muestreo o pesos.")

# Rango de datetime
df["datetime"] = pd.to_datetime(df["datetime"])
print(f"\nRango temporal de la muestra:")
print(f"  Inicio : {df['datetime'].min()}")
print(f"  Fin    : {df['datetime'].max()}")
print(f"  Granularidad: {df['datetime'].diff().mode()[0]}")

## 5. Conversión de tipos

| Columna | Tipo actual | Tipo objetivo | Justificación |
|---------|------------|---------------|---------------|
| `datetime` | object | datetime64[ns] | índice temporal |
| `id` | int64 | category | identificador de sensor, no valor numérico |
| `tipo_elem` | object | category | 2 valores: URB / M30 |
| `intensidad_trafico` | int64 | int32 | target, rango razonable (≤5412) |
| `ocupacion` | float64 | float32 | target, rango 0–100 |
| `carga` | int64 | int16 | target, rango 0–100 |
| `vmed` | float64 | float32 | velocidad media |
| `longitud` | float64 | float32 | coordenada |
| `latitud` | float64 | float32 | coordenada |
| `Dia_semana` | object | category (ordenado) | 7 valores fijos |
| `laborable / festivo / …` | object | category | 3 valores + NaN |
| `Tipo de Festivo` | object | category | pocos valores únicos |
| `Festividad` | object | category | baja cardinalidad |
| `precip_mm` | float64 | float32 | precipitación |
| `llueve` | float64 | **bool** | reconstruir desde intensidad_lluvia |
| `intensidad_lluvia` | object | category (ordenado) | ordinal: no_lluvia < leve < moderada < fuerte |
| `hay_accidente` | float64 | bool | flag binario |
| `tiempo_desde_accidente` | float64 | float32 | -1 = sin accidente |
| `heridos_leves` | float64 | int16 | conteo |
| `heridos_graves` | float64 | int16 | conteo |
| `victimas_mortales` | float64 | int16 | conteo |
| `nombre` | object | category | nombre del sensor (~baja cardinalidad única por id) |

In [ ]:
def convert_types(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica conversión de tipos optimizada al dataset de tráfico."""
    df = df.copy()

    # ── Datetime ──────────────────────────────────────────────────────────
    df["datetime"] = pd.to_datetime(df["datetime"])

    # ── Encoding: reparar mojibake en columnas de texto ───────────────────
    def fix_mojibake(s):
        if not isinstance(s, str):
            return s
        try:
            return s.encode("latin-1").decode("utf-8")
        except Exception:
            return s

    for col in ["nombre", "Dia_semana", "Festividad"]:
        df[col] = df[col].map(fix_mojibake)

    # ── Numéricas ─────────────────────────────────────────────────────────
    df["intensidad_trafico"] = df["intensidad_trafico"].astype("int32")
    df["ocupacion"]          = df["ocupacion"].astype("float32")
    df["carga"]              = df["carga"].astype("int16")
    df["vmed"]               = df["vmed"].astype("float32")
    df["longitud"]           = df["longitud"].astype("float32")
    df["latitud"]            = df["latitud"].astype("float32")
    df["precip_mm"]          = df["precip_mm"].astype("float32")
    df["tiempo_desde_accidente"] = df["tiempo_desde_accidente"].astype("float32")
    df["heridos_leves"]      = df["heridos_leves"].astype("int16")
    df["heridos_graves"]     = df["heridos_graves"].astype("int16")
    df["victimas_mortales"]  = df["victimas_mortales"].astype("int16")

    # ── Booleanas ─────────────────────────────────────────────────────────
    # 'llueve' se RECONSTRUYE desde intensidad_lluvia (la original está corrupta)
    df["llueve"]        = (df["intensidad_lluvia"] != "no_lluvia").astype("bool")
    df["hay_accidente"] = df["hay_accidente"].astype("bool")

    # ── Categóricas ordenadas ─────────────────────────────────────────────
    dias_order = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
    lluvia_order = ["no_lluvia", "leve", "moderada", "fuerte"]
    lab_order = ["laborable", "festivo", "domingo festivo"]

    df["Dia_semana"] = pd.Categorical(df["Dia_semana"], categories=dias_order, ordered=True)

    df["intensidad_lluvia"] = pd.Categorical(
        df["intensidad_lluvia"], categories=lluvia_order, ordered=True
    )
    df["laborable / festivo / domingo festivo"] = pd.Categorical(
        df["laborable / festivo / domingo festivo"], categories=lab_order, ordered=False
    )

    # ── Categóricas sin orden ─────────────────────────────────────────────
    for col in ["tipo_elem", "Tipo de Festivo", "Festividad", "nombre"]:
        df[col] = df[col].astype("category")

    df["id"] = df["id"].astype("category")

    return df


df_typed = convert_types(df)

print("Tipos después de la conversión:")
print(df_typed.dtypes)
print(f"\nMemoria antes : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Memoria después: {df_typed.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 6. Limpieza y tratamiento de nulos / valores erróneos

In [ ]:
def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpieza de Fase 1. NO crea nuevas features, solo:
      - Rellena nulos estructurales
      - Corrige valores incoherentes
      - Elimina columnas irrelevantes para el modelo
    """
    df = df.copy()

    # ── 6.1 Columnas a eliminar ───────────────────────────────────────────
    cols_drop = [c for c in ["longitud", "latitud"] if c in df.columns]
    df = df.drop(columns=cols_drop)
    print(f"Eliminadas: {cols_drop}")

    # ── 6.2 Nulos en columnas de festividad ───────────────────────────────
    # laborable/festivo/Tipo de Festivo/Festividad son NaN cuando el día
    # es laborable. Se rellena con categoría explícita.
    # fill_cat añade la categoría solo si aún no existe (evita ValueError
    # cuando convert_types ya la incluyó en lab_order).
    def fill_cat(series, fill_value):
        if fill_value not in series.cat.categories:
            series = series.cat.add_categories(fill_value)
        return series.fillna(fill_value)

    df["laborable / festivo / domingo festivo"] = fill_cat(
        df["laborable / festivo / domingo festivo"], "laborable"
    )
    df["Tipo de Festivo"] = fill_cat(df["Tipo de Festivo"], "Sin festivo")
    df["Festividad"]      = fill_cat(df["Festividad"],      "Sin festividad")
    print("Nulos de festividad rellenados con categoría explícita.")

    # ── 6.3 Nulos en vmed ────────────────────────────────────────────────
    # vmed nula ocurre en sensores M30 con fallo de medición.
    # Los sensores URB tienen vmed=0 por diseño (no miden velocidad).
    # Se usa forward-fill dentro de cada sensor (id) para M30.
    n_vmed_null = df["vmed"].isnull().sum()
    if n_vmed_null > 0:
        df = df.sort_values(["id", "datetime"])
        df["vmed"] = df.groupby("id", observed=True)["vmed"].transform(
            lambda x: x.ffill().bfill()
        )
        remaining = df["vmed"].isnull().sum()
        print(f"vmed: {n_vmed_null} nulos → forward/back fill por sensor. Restantes: {remaining}")

    # ── 6.4 Verificación final de nulos ──────────────────────────────────
    nulos_post = df.isnull().sum()
    nulos_post = nulos_post[nulos_post > 0]
    if nulos_post.empty:
        print("\nVerificación: sin nulos restantes.")
    else:
        print(f"\nNulos restantes:\n{nulos_post}")

    return df


df_clean = clean_dataset(df_typed)
print(f"\nShape final (muestra): {df_clean.shape}")
df_clean.head(3)

## 7. Carga completa con dtypes optimizados

El dataset completo tiene ~37.5M filas y 23 GB. Con los dtypes del paso 5 se reduce drásticamente la memoria. Se aplica en dos pasos:

1. Definir `dtype` dict para `pd.read_csv` (evita cargar todo como object)
2. Aplicar `convert_types` + `clean_dataset` sobre el DataFrame completo

> **Nota:** en un equipo con <32 GB RAM se recomienda procesar por chunks o usar Polars/Dask.

In [ ]:
# Dtypes especificados al leer → pandas no infiere, ahorra memoria y tiempo
READ_DTYPES = {
    "id":                                     "int32",
    "tipo_elem":                              "str",
    "intensidad_trafico":                     "int32",
    "ocupacion":                              "float32",
    "carga":                                  "int16",
    "vmed":                                   "float32",
    "nombre":                                 "str",
    "longitud":                               "float32",
    "latitud":                                "float32",
    "Dia_semana":                             "str",
    "laborable / festivo / domingo festivo":  "str",
    "Tipo de Festivo":                        "str",
    "Festividad":                             "str",
    "precip_mm":                              "float32",
    "llueve":                                 "float32",
    "intensidad_lluvia":                      "str",
    "hay_accidente":                          "float32",
    "tiempo_desde_accidente":                 "float32",
    "heridos_leves":                          "float32",
    "heridos_graves":                         "float32",
    "victimas_mortales":                      "float32",
}

print("Cargando dataset completo (~37.5M filas)... puede tardar varios minutos.")
print("Tip: si se queda sin RAM, usar CHUNK_SIZE=1_000_000 y procesar por lotes.\n")

df_full_raw = pd.read_csv(
    DATA_PATH,
    encoding="utf-8",
    dtype=READ_DTYPES,
    parse_dates=["datetime"],
    low_memory=False,
)

print(f"Dataset completo cargado: {df_full_raw.shape}")
print(f"Memoria raw: {df_full_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# Aplicar pipeline completo sobre el dataset full
df_full = convert_types(df_full_raw)
df_full = clean_dataset(df_full)

print(f"\nDataset limpio: {df_full.shape}")
print(f"Memoria final: {df_full.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"\nRango temporal completo:")
print(f"  Inicio : {df_full['datetime'].min()}")
print(f"  Fin    : {df_full['datetime'].max()}")
print(f"  Sensores únicos: {df_full['id'].nunique()}")
print(f"  tipo_elem:\n{df_full['tipo_elem'].value_counts()}")

In [ ]:
# Guardar en Parquet para carga rápida en fases posteriores
# Parquet comprime mucho mejor que CSV y conserva los dtypes
OUTPUT_PATH = "data/Trafico_con_accidentes_clean.parquet"
df_full.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")

import os
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Guardado en: {OUTPUT_PATH}")
print(f"Tamaño: {size_mb:.0f} MB  (vs ~6.4 GB del CSV original)")

## 8. Resumen de riesgos de Data Leakage

> **Data leakage** = incluir en las features información que en un escenario real de predicción NO estaría disponible en el momento de hacer la predicción.

### Variables con riesgo ALTO (información del futuro o post-evento)

| Columna | Riesgo | Explicación |
|---------|--------|-------------|
| `hay_accidente` | **ALTO** | Si el target es tráfico en el instante T, saber que hay un accidente en T es información del presente/futuro que un modelo real no tendría por adelantado. Solo usar si se predice con datos del pasado (T-k). |
| `tiempo_desde_accidente` | **ALTO** | Similar. El valor -1 o un número positivo solo se conoce cuando ya ocurrió el accidente. |
| `heridos_leves` / `heridos_graves` / `victimas_mortales` | **MUY ALTO** | Se conocen horas después del accidente. Nunca usar como features; solo como posibles targets de un modelo de severidad distinto. |

### Variables con riesgo MEDIO

| Columna | Riesgo | Explicación |
|---------|--------|-------------|
| `precip_mm` / `llueve` / `intensidad_lluvia` | **MEDIO** | Datos de precipitación del mismo intervalo. Son usables como feature si se dispone de la previsión meteorológica; si se usan datos observados en T, hay leakage pequeño pero aceptable en muchos contextos. Documentarlo. |

### Variables seguras (no leakage)

`Dia_semana`, `laborable / festivo / domingo festivo`, `Tipo de Festivo`, `Festividad`, `datetime`, `tipo_elem`, `id`, `nombre` → conocidas a priori.

---

### Recomendación para el TFG

```
Features input (pasado):   Dia_semana, laborable/festivo, tipo_elem,
                           precip_mm, intensidad_lluvia,
                           hay_accidente (T-k), tiempo_desde_accidente (T-k)

Targets a predecir (T):    intensidad_trafico, ocupacion, carga

Excluir del modelo:        heridos_leves, heridos_graves, victimas_mortales
                           (riesgo muy alto + desbalance >99.9%)
```

In [ ]:
# ── Resumen final de columnas limpias ────────────────────────────────────────
print("COLUMNAS FINALES DEL DATASET LIMPIO")
print("="*60)
summary = pd.DataFrame({
    "dtype":    df_clean.dtypes,
    "nulos":    df_clean.isnull().sum(),
    "únicos":   df_clean.nunique(),
    "muestra":  [str(df_clean[c].iloc[0])[:30] for c in df_clean.columns],
})
print(summary.to_string())

print(f"\nShape: {df_clean.shape}")
print("\nFASE 1 completada. Siguiente paso: Fase 2 — Feature Engineering.")

# FASE 2 — Feature Engineering
## TFG: Predicción de Tráfico Urbano (M30 + URB)

**Input:** `df_full` (resultado de Fase 1) · parquet en `data/Trafico_con_accidentes_clean.parquet`  
**Output:** `df_fe` · listo para Fase 3 (lags + LSTM/GRU)

**Modelos objetivo:** LSTM / GRU → las codificaciones cíclicas (sin/cos) son especialmente útiles porque evitan que la red vea hora=0 y hora=23 como extremos opuestos.

---
### Transformaciones
1. Variables temporales desde `datetime` (hora, día semana, fin de semana, mes + sin/cos)
2. Variable de calendario (`es_laborable`)
3. Meteorología (`lluvia_ord` ordinal 0-3, `llueve`, `precip_mm`)
4. Accidentes (`hay_accidente`, `accidente_reciente_1h/3h`, `tiempo_accidente_norm`)
5. Eliminación de columnas sin valor predictivo

---
### Índice
1. Carga desde parquet
2. Función `feature_engineering`
3. Aplicar y verificar
4. Guardar `df_fe`

In [ ]:
## 1. Carga desde parquet
# Si df_full ya está en memoria (ejecutado justo después de Fase 1), omitir esta celda.

CLEAN_PATH = "data/Trafico_con_accidentes_clean.parquet"

df_full = pd.read_parquet(CLEAN_PATH, engine="pyarrow")

print(f"Dataset cargado : {df_full.shape}")
print(f"Memoria         : {df_full.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Rango temporal  : {df_full['datetime'].min()} → {df_full['datetime'].max()}")
print(f"\nColumnas ({len(df_full.columns)}):")
print(df_full.dtypes.to_string())

In [ ]:
## 2. Función feature_engineering

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fase 2: genera features para LSTM/GRU y elimina columnas irrelevantes.
    No crea lags, no escala, no hace split.
    """
    df = df.copy()

    # ─────────────────────────────────────────────────────────────────────
    # 1. VARIABLES TEMPORALES DESDE datetime
    # ─────────────────────────────────────────────────────────────────────

    # Hora del día (0-23) — útil para árboles; para LSTM se usa slot_sin/cos
    df["hora"] = df["datetime"].dt.hour.astype("int8")

    # Día de la semana reconstruido desde datetime (0=lun … 6=dom)
    # Se ignora la columna Dia_semana original (mojibake + redundante)
    df["dia_semana_num"] = df["datetime"].dt.dayofweek.astype("int8")

    # Fin de semana
    df["es_fin_de_semana"] = (df["dia_semana_num"] >= 5).astype("int8")

    # Mes (1-12)
    df["mes"] = df["datetime"].dt.month.astype("int8")

    # ── Codificación cíclica (imprescindible para LSTM/GRU) ──────────────
    # Sin/cos evitan la discontinuidad en los extremos del ciclo
    # (hora=23 y hora=0 son adyacentes, pero en escala lineal parecen opuestos)

    # Slot de 15 min dentro del día (0-95)
    slot = df["datetime"].dt.hour * 4 + df["datetime"].dt.minute // 15
    df["slot_sin"] = np.sin(2 * np.pi * slot / 96).astype("float32")
    df["slot_cos"] = np.cos(2 * np.pi * slot / 96).astype("float32")

    # Día de la semana cíclico
    df["dia_sem_sin"] = np.sin(2 * np.pi * df["dia_semana_num"] / 7).astype("float32")
    df["dia_sem_cos"] = np.cos(2 * np.pi * df["dia_semana_num"] / 7).astype("float32")

    # Mes cíclico (estacionalidad anual)
    df["mes_sin"] = np.sin(2 * np.pi * (df["mes"] - 1) / 12).astype("float32")
    df["mes_cos"] = np.cos(2 * np.pi * (df["mes"] - 1) / 12).astype("float32")

    # Eliminar Dia_semana original (reemplazada por dia_semana_num + cíclicas)
    df = df.drop(columns=["Dia_semana"], errors="ignore")

    # ─────────────────────────────────────────────────────────────────────
    # 2. VARIABLE DE CALENDARIO
    # ─────────────────────────────────────────────────────────────────────

    # es_laborable: 1 = día laborable, 0 = festivo o domingo festivo
    # La distinción festivo vs domingo-festivo no aporta señal adicional
    # (ambos tienen patrón de tráfico similar al fin de semana)
    lab_col = "laborable / festivo / domingo festivo"
    df["es_laborable"] = (df[lab_col] == "laborable").astype("int8")
    df = df.drop(columns=[lab_col], errors="ignore")

    # ─────────────────────────────────────────────────────────────────────
    # 3. VARIABLES METEOROLÓGICAS
    # ─────────────────────────────────────────────────────────────────────

    # intensidad_lluvia → ordinal 0-3
    # Preserva el orden real sin perder información (no one-hot)
    lluvia_map = {"no_lluvia": 0, "leve": 1, "moderada": 2, "fuerte": 3}
    df["lluvia_ord"] = (
        df["intensidad_lluvia"].map(lluvia_map).fillna(0).astype("int8")
    )

    # llueve: bool → int8 (compatibilidad con MinMaxScaler y tensores)
    df["llueve"] = df["llueve"].astype("int8")

    # precip_mm: magnitud continua de precipitación (complementa lluvia_ord)
    # lluvia_ord dice "leve/moderada/fuerte"; precip_mm dice "exactamente cuánto"

    # Eliminar intensidad_lluvia (reemplazada por lluvia_ord)
    df = df.drop(columns=["intensidad_lluvia"], errors="ignore")

    # ─────────────────────────────────────────────────────────────────────
    # 4. VARIABLES DE ACCIDENTES
    # ─────────────────────────────────────────────────────────────────────

    # hay_accidente: bool → int8
    df["hay_accidente"] = df["hay_accidente"].astype("int8")

    # tiempo_desde_accidente está en slots de 15 min.
    # Centinelas: -1 y 9999 = "sin accidente reciente"
    tda = df["tiempo_desde_accidente"].copy()
    sin_acc = (tda < 0) | (tda >= 9999)

    # accidente_reciente_1h: accidente en los últimos 4 slots (60 min)
    df["accidente_reciente_1h"] = ((~sin_acc) & (tda <= 4)).astype("int8")

    # accidente_reciente_3h: accidente en los últimos 12 slots (3 h)
    # 3 h coincide con la ventana del LSTM (window=12 timesteps × 15 min)
    df["accidente_reciente_3h"] = ((~sin_acc) & (tda <= 12)).astype("int8")

    # tiempo_accidente_norm: variable continua de proximidad temporal
    # 0.0 = accidente ahora mismo · 1.0 = sin accidente (o hace > 3 h)
    tda_capped = tda.clip(lower=0, upper=12)
    tda_norm = tda_capped / 12.0
    tda_norm[sin_acc] = 1.0
    df["tiempo_accidente_norm"] = tda_norm.astype("float32")

    df = df.drop(columns=["tiempo_desde_accidente"], errors="ignore")

    # ─────────────────────────────────────────────────────────────────────
    # 5. ELIMINAR COLUMNAS SIN VALOR PREDICTIVO
    # ─────────────────────────────────────────────────────────────────────

    cols_drop = [
        "nombre",           # texto libre, redundante con id (category)
        "Tipo de Festivo",  # alta cardinalidad + no generaliza a nuevos años
        "Festividad",       # ídem
        "heridos_leves",    # leakage muy alto + desbalance >99.9%
        "heridos_graves",   # ídem
        "victimas_mortales",# ídem (se conocen horas después del accidente)
        "vmed",             # URB = siempre 0 (no mide velocidad); se descarta
                            # para no añadir ruido al modelo conjunto
    ]
    df = df.drop(columns=[c for c in cols_drop if c in df.columns])

    return df

In [ ]:
## 3. Aplicar y verificar

df_fe = feature_engineering(df_full)

# ── Resumen de columnas ───────────────────────────────────────────────────────
num_cols = df_fe.select_dtypes(include="number").columns

summary_fe = pd.DataFrame({
    "dtype":  df_fe.dtypes,
    "nulos":  df_fe.isnull().sum(),
    "únicos": df_fe.nunique(),
    "min":    df_fe[num_cols].min().reindex(df_fe.columns),
    "max":    df_fe[num_cols].max().reindex(df_fe.columns),
})

print("COLUMNAS FINALES — FASE 2")
print("=" * 70)
print(summary_fe.to_string())
print(f"\nShape total : {df_fe.shape}")
print(f"Memoria     : {df_fe.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# ── Sin nulos ────────────────────────────────────────────────────────────────
nulos_restantes = df_fe.isnull().sum()
nulos_restantes = nulos_restantes[nulos_restantes > 0]
if nulos_restantes.empty:
    print("\n✓ Sin nulos en df_fe.")
else:
    print(f"\nATENCIÓN — Nulos detectados:\n{nulos_restantes}")

# ── Rango cíclicas ───────────────────────────────────────────────────────────
print("\nRango features cíclicas (esperado: [-1, 1]):")
for col in ["slot_sin", "slot_cos", "dia_sem_sin", "dia_sem_cos", "mes_sin", "mes_cos"]:
    mn, mx = df_fe[col].min(), df_fe[col].max()
    print(f"  {col:15s}  [{mn:.4f}, {mx:.4f}]")

# ── Desbalance de accidentes ─────────────────────────────────────────────────
print("\nDistribución variables de accidente:")
for col in ["hay_accidente", "accidente_reciente_1h", "accidente_reciente_3h"]:
    n1 = df_fe[col].sum()
    pct = n1 / len(df_fe) * 100
    print(f"  {col:25s}: {n1:>8,}  ({pct:.3f}%)")

In [ ]:
## 4. Guardar df_fe para Fase 3

FE_PATH = "data/Trafico_fe.parquet"
df_fe.to_parquet(FE_PATH, index=False, engine="pyarrow", compression="snappy")

import os
size_mb = os.path.getsize(FE_PATH) / 1e6
print(f"Guardado en : {FE_PATH}")
print(f"Tamaño      : {size_mb:.0f} MB")
print(f"Shape       : {df_fe.shape}")
print("\nColumnas finales:")
for col in df_fe.columns:
    print(f"  {col:25s}  {str(df_fe[col].dtype):10s}")
print("\nFASE 2 completada. Siguiente: Fase 3 — Lags + Split cronológico + LSTM/GRU.")

# FASE 3 — Variables LAG, Medias Móviles y Split Temporal
## TFG: Predicción de Tráfico Urbano (M30 + URB)

**Input:** `data/Trafico_fe.parquet` (resultado de Fase 2)  
**Output:** `data/train.parquet`, `data/val.parquet`, `data/test.parquet`

---

### Por qué los lags son imprescindibles

El tráfico en el instante **T** depende fuertemente de los instantes anteriores:
- Un atasco que empieza en T−1 (hace 15 min) ya está presente en T.
- La hora punta de la mañana dura varias horas: `intensidad[T]` predice `intensidad[T+1]`.

Sin lags, el modelo recibe **solo el instante actual** y no tiene "memoria": trata cada fila como independiente aunque sean consecutivas del mismo sensor. Con lags, convertimos la serie temporal en un problema supervisado estándar donde el pasado es parte del vector de features.

### Por qué groupby(id) antes de aplicar shift
Los sensores son series independientes. Si no agrupamos, el lag_1 del primer instante del sensor B sería el último instante del sensor A — datos de otra calle. `groupby('id').shift(k)` garantiza que los lags solo cruzan filas del mismo sensor.

### Split temporal estricto
El corte se hace por timestamp global (70 % / 15 % / 15 % de las fechas únicas).  
Todos los sensores tienen exactamente el mismo horizonte de entrenamiento.  
`shuffle=False` siempre — barajar introduciría leakage del futuro al pasado.

---
### Índice
1. Carga desde parquet
2. Lags (t−1 … t−4) por sensor
3. Medias móviles (ventana 4 y 8) por sensor
4. Eliminación de NaN y definición de targets / features
5. Split temporal 70 / 15 / 15
6. Guardar y resumen final

In [ ]:
## 1. Carga desde parquet
# Si df_fe ya está en memoria (ejecutado justo después de Fase 2), omitir esta celda.

import pandas as pd
import numpy as np
import os

FE_PATH = "data/Trafico_fe.parquet"

df_fe = pd.read_parquet(FE_PATH, engine="pyarrow")

print(f"Dataset cargado : {df_fe.shape}")
print(f"Memoria         : {df_fe.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Rango temporal  : {df_fe['datetime'].min()} → {df_fe['datetime'].max()}")
print(f"Sensores únicos : {df_fe['id'].nunique()}")
print(f"\nColumnas ({len(df_fe.columns)}): {list(df_fe.columns)}")

In [ ]:
## 2 & 3. Lags (t-1 … t-4) + Medias móviles (ventana 4 y 8) por sensor

TARGETS = ["intensidad_trafico", "ocupacion", "carga"]
LAG_STEPS = [1, 2, 3, 4]
ROLL_WINDOWS = [4, 8]

# ── Orden temporal estricto por sensor antes de calcular lags ────────────────
# CRÍTICO: sin este sort, shift(k) mezclaría instantes desordenados.
print("Ordenando por (id, datetime)...")
df = df_fe.sort_values(["id", "datetime"]).reset_index(drop=True)
print(f"Shape tras sort: {df.shape}")

# ── Lags: t-1, t-2, t-3, t-4 ────────────────────────────────────────────────
# groupby('id') garantiza que los lags NO cruzan sensores distintos.
# shift(k) desplaza k posiciones hacia adelante dentro del grupo.
print("\nCreando lags...")
for target in TARGETS:
    g = df.groupby("id", observed=True)[target]
    for k in LAG_STEPS:
        col_name = f"{target}_lag{k}"
        df[col_name] = g.shift(k).astype("float32")
        print(f"  {col_name}")

# ── Medias móviles: ventana 4 (1 h) y ventana 8 (2 h) ───────────────────────
# Se usa shift(1) ANTES del rolling para que la ventana sea [t-w … t-1].
# Sin el shift(1), la ventana incluiría t → leakage del target actual.
# min_periods=w: la celda queda NaN hasta completar la ventana completa.
print("\nCreando medias móviles...")
for target in TARGETS:
    g = df.groupby("id", observed=True)[target]
    for w in ROLL_WINDOWS:
        col_name = f"{target}_roll{w}"
        df[col_name] = (
            g.transform(lambda x: x.shift(1).rolling(w, min_periods=w).mean())
            .astype("float32")
        )
        print(f"  {col_name}")

lag_roll_cols = (
    [f"{t}_lag{k}" for t in TARGETS for k in LAG_STEPS] +
    [f"{t}_roll{w}" for t in TARGETS for w in ROLL_WINDOWS]
)

print(f"\nNuevas columnas añadidas : {len(lag_roll_cols)}")
print(f"Shape con lags/rolls    : {df.shape}")
print(f"\nNaN generados por ventana roll8 (esperado ≈ 8 × n_sensores):")
for col in lag_roll_cols:
    n_nan = df[col].isna().sum()
    if n_nan > 0:
        print(f"  {col:35s}: {n_nan:,} NaN")

In [ ]:
## 4. Eliminación de NaN y definición de targets / features

# ── Eliminar filas con NaN en las columnas lag/roll ──────────────────────────
n_antes = len(df)
df_lag = df.dropna(subset=lag_roll_cols).reset_index(drop=True)
n_despues = len(df_lag)
print(f"Filas antes  : {n_antes:,}")
print(f"Filas después: {n_despues:,}")
print(f"Eliminadas   : {n_antes - n_despues:,}  ({(n_antes-n_despues)/n_antes*100:.4f} %)")

# ── Codificar tipo_elem como numérico ─────────────────────────────────────────
# tipo_elem tiene valores 'URB' y 'M30'. Se codifica como int8 (0/1)
# para que sea compatible con float32. No se filtra ni separa nada.
df_lag["tipo_elem"] = (df_lag["tipo_elem"] == "M30").astype("int8")
print(f"\ntipo_elem codificado → 0=URB, 1=M30")
print(df_lag["tipo_elem"].value_counts().to_string())

# ── Definición de targets ────────────────────────────────────────────────────
TARGET_COLS = TARGETS   # ["intensidad_trafico", "ocupacion", "carga"]

# ── Definición de features ───────────────────────────────────────────────────
EXCLUDE_COLS = {"datetime", "id"} | set(TARGET_COLS)
FEATURE_COLS = [c for c in df_lag.columns if c not in EXCLUDE_COLS]

print(f"\nTargets  ({len(TARGET_COLS)}): {TARGET_COLS}")
print(f"\nFeatures ({len(FEATURE_COLS)}):")
for col in FEATURE_COLS:
    print(f"  {col:35s}  {str(df_lag[col].dtype):10s}")

# ── Verificación: todas las features son numéricas ───────────────────────────
non_numeric = [c for c in FEATURE_COLS if not pd.api.types.is_numeric_dtype(df_lag[c])]
if non_numeric:
    print(f"\nATENCIÓN — columnas no numéricas en FEATURE_COLS: {non_numeric}")
else:
    print(f"\nTodas las features son numéricas.")

# ── Verificación de nulos ─────────────────────────────────────────────────────
nulos = df_lag[FEATURE_COLS + TARGET_COLS].isnull().sum()
nulos = nulos[nulos > 0]
if nulos.empty:
    print("Sin nulos en features ni targets.")
else:
    print(f"\nATENCIÓN — Nulos detectados:\n{nulos}")

In [ ]:
## 5. Split temporal 70 / 15 / 15 (sin shuffle)

# ── Corte por timestamps únicos ──────────────────────────────────────────────
# Se calcula el percentil 70 y 85 de las fechas ÚNICAS del dataset.
# Todos los sensores quedan con el mismo horizonte de train/val/test.
# NO se usa train_test_split → evita mezclar instantes del futuro en el pasado.

timestamps_sorted = np.sort(df_lag["datetime"].unique())
n_ts = len(timestamps_sorted)

corte_70 = timestamps_sorted[int(0.70 * n_ts)]
corte_85 = timestamps_sorted[int(0.85 * n_ts)]

print(f"Timestamps únicos : {n_ts:,}")
print(f"Corte 70 %        : {corte_70}")
print(f"Corte 85 %        : {corte_85}")

train_df = df_lag[df_lag["datetime"] <  corte_70].reset_index(drop=True)
val_df   = df_lag[(df_lag["datetime"] >= corte_70) & (df_lag["datetime"] < corte_85)].reset_index(drop=True)
test_df  = df_lag[df_lag["datetime"] >= corte_85].reset_index(drop=True)

print(f"\nSplit:")
print(f"  Train : {len(train_df):>10,} filas  ({len(train_df)/len(df_lag)*100:.1f} %)"
      f"  [{train_df['datetime'].min()} → {train_df['datetime'].max()}]")
print(f"  Val   : {len(val_df):>10,} filas  ({len(val_df)/len(df_lag)*100:.1f} %)"
      f"  [{val_df['datetime'].min()} → {val_df['datetime'].max()}]")
print(f"  Test  : {len(test_df):>10,} filas  ({len(test_df)/len(df_lag)*100:.1f} %)"
      f"  [{test_df['datetime'].min()} → {test_df['datetime'].max()}]")

# ── Verificación: no hay solapamiento ────────────────────────────────────────
overlap_tv = set(train_df["datetime"]) & set(val_df["datetime"])
overlap_vt = set(val_df["datetime"])  & set(test_df["datetime"])
print(f"\nSolapamiento train/val   : {len(overlap_tv)} timestamps (esperado 0)")
print(f"Solapamiento val/test    : {len(overlap_vt)} timestamps (esperado 0)")

# ── Matrices X / y ────────────────────────────────────────────────────────────
# Útiles directamente para XGBoost, RandomForest, etc.
# Para LSTM/GRU se usarán los DataFrames completos con reshape posterior.
X_train = train_df[FEATURE_COLS].values.astype("float32")
y_train = train_df[TARGET_COLS].values.astype("float32")

X_val   = val_df[FEATURE_COLS].values.astype("float32")
y_val   = val_df[TARGET_COLS].values.astype("float32")

X_test  = test_df[FEATURE_COLS].values.astype("float32")
y_test  = test_df[TARGET_COLS].values.astype("float32")

print(f"\nX_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}   y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}   y_test  : {y_test.shape}")

In [ ]:
## 6. Guardar splits y resumen final

TRAIN_PATH = "data/train.parquet"
VAL_PATH   = "data/val.parquet"
TEST_PATH  = "data/test.parquet"

train_df.to_parquet(TRAIN_PATH, index=False, engine="pyarrow", compression="snappy")
val_df.to_parquet(VAL_PATH,     index=False, engine="pyarrow", compression="snappy")
test_df.to_parquet(TEST_PATH,   index=False, engine="pyarrow", compression="snappy")

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    mb = os.path.getsize(path) / 1e6
    print(f"Guardado: {path:30s}  {mb:.0f} MB")

# ── Resumen final ─────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("RESUMEN FASE 3")
print(f"{'='*65}")
print(f"Filas totales (sin NaN)  : {len(df_lag):,}")
print(f"  Train                  : {len(train_df):,}  ({len(train_df)/len(df_lag)*100:.1f} %)")
print(f"  Validation             : {len(val_df):,}  ({len(val_df)/len(df_lag)*100:.1f} %)")
print(f"  Test                   : {len(test_df):,}  ({len(test_df)/len(df_lag)*100:.1f} %)")
print(f"\nDimensión X              : {X_train.shape[1]} features")
print(f"Dimensión y              : {y_train.shape[1]} targets")
print(f"\nFeatures de lag          : {sum(1 for c in FEATURE_COLS if 'lag' in c)}")
print(f"Features de rolling mean : {sum(1 for c in FEATURE_COLS if 'roll' in c)}")
print(f"Features temporales/otros: {sum(1 for c in FEATURE_COLS if 'lag' not in c and 'roll' not in c)}")
print(f"\nColumnas finales del dataset ({len(df_lag.columns)}):")
for col in df_lag.columns:
    print(f"  {col:35s}  {str(df_lag[col].dtype)}")
print(f"\nFASE 3 completada. Siguiente: Fase 4 — Normalización + Modelos (XGBoost / LSTM / GRU).")

# FASE 4 — Modelos de Predicción: Baseline y XGBoost
## TFG: Predicción de Tráfico Urbano (M30 + URB)

**Input:** `data/train.parquet`, `data/val.parquet`, `data/test.parquet`  
**Targets:** `intensidad_trafico`, `ocupacion`, `carga`

---

### Estrategia
- **Baseline**: predicción trivial usando el valor en t−1 (lag_1). Marca el mínimo que cualquier modelo debe superar.
- **XGBoost**: 3 modelos independientes (uno por target). XGBoost no soporta multi-output nativo; entrenar por separado permite además ajustar hiperparámetros por target.
- **GPU**: se usa `device='cuda'` con `tree_method='hist'` para acelerar el entrenamiento sobre ~37.5M filas.

### Métricas
- **MAE** (Mean Absolute Error): error medio en las mismas unidades que el target. Fácil de interpretar.
- **RMSE** (Root Mean Squared Error): penaliza más los errores grandes. Más sensible a picos de tráfico.

---
### Índice
1. Carga y preparación de datos
2. Baseline (lag_1)
3. XGBoost — entrenamiento
4. Evaluación comparativa (val + test)
5. Importancia de variables

In [ ]:
## 1. Carga y preparación de datos

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

TARGET_COLS  = ["intensidad_trafico", "ocupacion", "carga"]
DROP_FROM_X  = set(TARGET_COLS) | {"datetime", "id"}

train_df = pd.read_parquet("data/train.parquet", engine="pyarrow")
val_df   = pd.read_parquet("data/val.parquet",   engine="pyarrow")
test_df  = pd.read_parquet("data/test.parquet",  engine="pyarrow")

FEATURE_COLS = [c for c in train_df.columns if c not in DROP_FROM_X]

X_train = train_df[FEATURE_COLS].values.astype("float32")
y_train = train_df[TARGET_COLS].values.astype("float32")

X_val   = val_df[FEATURE_COLS].values.astype("float32")
y_val   = val_df[TARGET_COLS].values.astype("float32")

X_test  = test_df[FEATURE_COLS].values.astype("float32")
y_test  = test_df[TARGET_COLS].values.astype("float32")

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}   y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}   y_test  : {y_test.shape}")
print(f"\nFeatures ({len(FEATURE_COLS)}):")
for i, c in enumerate(FEATURE_COLS):
    print(f"  [{i:02d}] {c}")

In [ ]:
## 2. Baseline: predicción = valor en t-1 (lag_1)
# El baseline más simple posible: "el tráfico en T es igual al de T-1".
# Si el modelo no supera esto, no justifica su complejidad.

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

baseline_metrics = {}

for split_name, df_split, y_true in [
    ("val",  val_df,  y_val),
    ("test", test_df, y_test),
]:
    preds = {}
    for i, target in enumerate(TARGET_COLS):
        lag_col = f"{target}_lag1"
        y_pred  = df_split[lag_col].values.astype("float32")
        mae  = mean_absolute_error(y_true[:, i], y_pred)
        rms  = rmse(y_true[:, i], y_pred)
        preds[target] = y_pred
        baseline_metrics[(split_name, target)] = {"MAE": mae, "RMSE": rms}

print("BASELINE (lag_1) — Resultados")
print("=" * 55)
for split_name in ["val", "test"]:
    print(f"\n  {split_name.upper()}")
    for target in TARGET_COLS:
        m = baseline_metrics[(split_name, target)]
        print(f"    {target:25s}  MAE={m['MAE']:8.3f}  RMSE={m['RMSE']:8.3f}")

In [ ]:
## 3. XGBoost — entrenamiento (3 modelos, uno por target)

from xgboost import XGBRegressor

# ── Detección de GPU con comprobación de memoria libre ────────────────────────
# Solo usamos CUDA si hay ≥500 MB libres. Si TensorFlow (Fase 5) ya reservó VRAM
# en la misma sesión, XGBoost falla con OOM aunque la GPU exista.
XGB_DEVICE = "cpu"
try:
    import subprocess, re
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
        stderr=subprocess.DEVNULL,
    ).decode().strip()
    free_mb = min(int(x) for x in out.splitlines() if x.strip().isdigit())
    if free_mb >= 500:
        XGB_DEVICE = "cuda"
        print(f"GPU detectada — {free_mb} MB libres → device='cuda'")
    else:
        print(f"GPU con solo {free_mb} MB libres (TF puede estar ocupando VRAM) → device='cpu'")
except Exception:
    print("nvidia-smi no disponible → device='cpu'")

XGB_PARAMS = dict(
    n_estimators          = 1000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 10,   # evita sobreajuste en zonas de bajo tráfico
    tree_method           = "hist",
    max_bin               = 128,  # reduce VRAM: 128 bins vs 256 por defecto
    device                = XGB_DEVICE,
    early_stopping_rounds = 20,
    eval_metric           = "mae",
    verbosity             = 0,
    random_state          = 42,
)

xgb_models = {}

for i, target in enumerate(TARGET_COLS):
    print(f"\nEntrenando XGBoost → {target} ...")
    model = XGBRegressor(**XGB_PARAMS)
    try:
        model.fit(
            X_train, y_train[:, i],
            eval_set=[(X_val, y_val[:, i])],
            verbose=100,
        )
    except Exception as e:
        # OOM en GPU (bad_alloc / cudaErrorMemoryAllocation) → reintento en CPU
        if "bad_alloc" in str(e) or "out of memory" in str(e).lower():
            print(f"  OOM en GPU ({e.__class__.__name__}), reintentando en CPU...")
            params_cpu = {**XGB_PARAMS, "device": "cpu"}
            model = XGBRegressor(**params_cpu)
            model.fit(
                X_train, y_train[:, i],
                eval_set=[(X_val, y_val[:, i])],
                verbose=100,
            )
        else:
            raise

    xgb_models[target] = model
    print(f"  Mejor iteración: {model.best_iteration}")

print("\nEntrenamiento completado.")


In [ ]:
## 4. Evaluación comparativa — Baseline vs XGBoost

xgb_metrics = {}

for split_name, X_sp, y_sp in [
    ("val",  X_val,  y_val),
    ("test", X_test, y_test),
]:
    for i, target in enumerate(TARGET_COLS):
        y_pred = xgb_models[target].predict(X_sp)
        xgb_metrics[(split_name, target)] = {
            "MAE":  mean_absolute_error(y_sp[:, i], y_pred),
            "RMSE": rmse(y_sp[:, i], y_pred),
        }

# ── Tabla comparativa ─────────────────────────────────────────────────────────
print("COMPARATIVA BASELINE vs XGBOOST")
print("=" * 75)
header = f"{'':27s}  {'--- BASELINE ---':^22s}  {'--- XGBOOST ---':^22s}"
print(header)
print(f"{'Split + Target':27s}  {'MAE':>10s}  {'RMSE':>10s}  {'MAE':>10s}  {'RMSE':>10s}")
print("-" * 75)

for split_name in ["val", "test"]:
    print(f"\n  {split_name.upper()}")
    for target in TARGET_COLS:
        bm = baseline_metrics[(split_name, target)]
        xm = xgb_metrics[(split_name, target)]
        mae_imp  = (bm["MAE"]  - xm["MAE"])  / bm["MAE"]  * 100
        rmse_imp = (bm["RMSE"] - xm["RMSE"]) / bm["RMSE"] * 100
        print(
            f"    {target:23s}  "
            f"{bm['MAE']:10.3f}  {bm['RMSE']:10.3f}  "
            f"{xm['MAE']:10.3f}  {xm['RMSE']:10.3f}  "
            f"  ↓MAE {mae_imp:+.1f}%  ↓RMSE {rmse_imp:+.1f}%"
        )

# ── Gráfico barras comparativo ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Baseline vs XGBoost — MAE y RMSE por target", fontsize=14)

for col, target in enumerate(TARGET_COLS):
    for row, metric in enumerate(["MAE", "RMSE"]):
        ax = axes[row, col]
        vals = {
            "Baseline\n(val)":  baseline_metrics[("val",  target)][metric],
            "XGBoost\n(val)":   xgb_metrics[("val",  target)][metric],
            "Baseline\n(test)": baseline_metrics[("test", target)][metric],
            "XGBoost\n(test)":  xgb_metrics[("test", target)][metric],
        }
        colors = ["#90A4AE", "#1565C0", "#90A4AE", "#1565C0"]
        bars = ax.bar(list(vals.keys()), list(vals.values()), color=colors, edgecolor="white")
        ax.set_title(f"{target}\n{metric}", fontsize=11)
        ax.set_ylabel(metric)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h * 1.01,
                    f"{h:.2f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("outputs/fase4_comparativa.png", dpi=120, bbox_inches="tight")
plt.show()
print("Gráfico guardado: outputs/fase4_comparativa.png")

In [ ]:
## 5. Importancia de variables

TOP_N = 20

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
fig.suptitle(f"XGBoost — Top {TOP_N} features por target", fontsize=14)

importances_df = {}

for ax, target in zip(axes, TARGET_COLS):
    model = xgb_models[target]
    imp   = model.feature_importances_

    df_imp = (
        pd.DataFrame({"feature": FEATURE_COLS, "importance": imp})
        .sort_values("importance", ascending=False)
        .head(TOP_N)
        .reset_index(drop=True)
    )
    importances_df[target] = df_imp

    # Colores: lags en azul, rolls en naranja, resto en gris
    colors = []
    for f in df_imp["feature"]:
        if "lag" in f:
            colors.append("#1565C0")
        elif "roll" in f:
            colors.append("#E65100")
        else:
            colors.append("#546E7A")

    ax.barh(df_imp["feature"][::-1], df_imp["importance"][::-1],
            color=colors[::-1], edgecolor="white")
    ax.set_title(target, fontsize=12)
    ax.set_xlabel("Importance (gain)")
    ax.tick_params(axis="y", labelsize=9)

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#1565C0", label="lag features"),
    Patch(facecolor="#E65100", label="rolling mean features"),
    Patch(facecolor="#546E7A", label="otras features"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig("outputs/fase4_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Gráfico guardado: outputs/fase4_feature_importance.png")

# ── Top 5 por target en texto ──────────────────────────────────────────────────
print("\nTOP 5 FEATURES POR TARGET")
print("=" * 55)
for target in TARGET_COLS:
    print(f"\n  {target}")
    for _, row in importances_df[target].head(5).iterrows():
        print(f"    {row['feature']:35s}  {row['importance']:.6f}")

# ── Features nunca usadas (importance = 0) ────────────────────────────────────
print("\nFEATURES CON IMPORTANCIA = 0 EN TODOS LOS MODELOS:")
zero_all = [
    f for f in FEATURE_COLS
    if all(xgb_models[t].feature_importances_[FEATURE_COLS.index(f)] == 0
           for t in TARGET_COLS)
]
if zero_all:
    for f in zero_all:
        print(f"  {f}")
else:
    print("  Ninguna — todas contribuyen en al menos un modelo.")

print("\nFASE 4 completada.")

# FASE 5 — Modelo LSTM para Series Temporales
## TFG: Predicción de Tráfico Urbano (M30 + URB)

**Input:** `data/train.parquet`, `data/val.parquet`, `data/test.parquet`  
**Targets:** `intensidad_trafico`, `ocupacion`, `carga`

---

### Estrategia de secuencias

En lugar de recargar el CSV completo, se reconstruyen secuencias de 4 pasos temporales
directamente desde las columnas lag ya calculadas en Fase 3:

```
Paso t-4: [intensidad_lag4, ocupacion_lag4, carga_lag4, features_contexto]
Paso t-3: [intensidad_lag3, ocupacion_lag3, carga_lag3, features_contexto]
Paso t-2: [intensidad_lag2, ocupacion_lag2, carga_lag2, features_contexto]
Paso t-1: [intensidad_lag1, ocupacion_lag1, carga_lag1, features_contexto]
         ──────────────────────────────────────────────────────────────────
Salida t: [intensidad_trafico, ocupacion, carga]
```

4 pasos × 15 minutos = **1 hora de historial** por muestra.  
Las `features_contexto` (temporales, meteorológicas, accidentes, rolls) se repiten en cada paso porque representan el estado en T, no el histórico.

### Arquitectura
```
Input(4, n_features) → LSTM(128) → Dropout → LSTM(64) → Dropout → Dense(32) → Dense(3)
```

### GPU y precisión mixta
Consistente con el resto del proyecto: `CUDA_VISIBLE_DEVICES=0`, `mixed_float16`.

---
### Índice
1. Preparación de secuencias y normalización
2. Definición del modelo LSTM
3. Entrenamiento
4. Evaluación y comparativa XGBoost vs LSTM

In [ ]:
## 1. Preparación de secuencias y normalización

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import mixed_precision
from sklearn.preprocessing import StandardScaler

mixed_precision.set_global_policy("mixed_float16")

TARGET_COLS = ["intensidad_trafico", "ocupacion", "carga"]
SEQ_STEPS   = [4, 3, 2, 1]   # t-4 → t-1, del más antiguo al más reciente
SEQ_LEN     = len(SEQ_STEPS)

# Features de contexto que se repiten en cada paso de la secuencia
CONTEXT_COLS = [
    "slot_sin", "slot_cos",
    "dia_sem_sin", "dia_sem_cos",
    "mes_sin", "mes_cos",
    "es_laborable",
    "lluvia_ord", "precip_mm",
    "hay_accidente", "accidente_reciente_1h", "tiempo_accidente_norm",
    "tipo_elem",
    "intensidad_trafico_roll4", "intensidad_trafico_roll8",
    "ocupacion_roll4",          "ocupacion_roll8",
    "carga_roll4",              "carga_roll8",
]

def build_sequences(df):
    """Construye tensor (n, SEQ_LEN, n_features) desde el DataFrame con columnas lag."""
    context = df[CONTEXT_COLS].values.astype("float32")   # (n, n_ctx)
    steps = []
    for k in SEQ_STEPS:
        traffic_k = np.column_stack([
            df[f"intensidad_trafico_lag{k}"].values,
            df[f"ocupacion_lag{k}"].values,
            df[f"carga_lag{k}"].values,
        ]).astype("float32")                               # (n, 3)
        steps.append(np.concatenate([traffic_k, context], axis=1))
    return np.stack(steps, axis=1)                         # (n, SEQ_LEN, 3+n_ctx)

# Cargar parquets (si ya están en memoria desde Fase 4, se reusan)
if "train_df" not in dir():
    train_df = pd.read_parquet("data/train.parquet", engine="pyarrow")
    val_df   = pd.read_parquet("data/val.parquet",   engine="pyarrow")
    test_df  = pd.read_parquet("data/test.parquet",  engine="pyarrow")

print("Construyendo secuencias...")
X_seq_train = build_sequences(train_df)
X_seq_val   = build_sequences(val_df)
X_seq_test  = build_sequences(test_df)

y_train = train_df[TARGET_COLS].values.astype("float32")
y_val   = val_df[TARGET_COLS].values.astype("float32")
y_test  = test_df[TARGET_COLS].values.astype("float32")

N_FEATURES = X_seq_train.shape[2]
print(f"X_seq_train : {X_seq_train.shape}   y_train : {y_train.shape}")
print(f"X_seq_val   : {X_seq_val.shape}   y_val   : {y_val.shape}")
print(f"X_seq_test  : {X_seq_test.shape}   y_test  : {y_test.shape}")
print(f"Features por paso: {N_FEATURES}  ({SEQ_LEN} pasos × 15 min = 1 h de historial)")

# ── Normalización ─────────────────────────────────────────────────────────────
# StandardScaler ajustado SOLO sobre train (sin contaminar val/test).
# Se aplica sobre la dimensión de features: reshape (n, 4, f) → (n*4, f) → vuelta.
scaler_X = StandardScaler()
shape_train = X_seq_train.shape
X_seq_train_n = scaler_X.fit_transform(
    X_seq_train.reshape(-1, N_FEATURES)).reshape(shape_train)
X_seq_val_n  = scaler_X.transform(
    X_seq_val.reshape(-1, N_FEATURES)).reshape(X_seq_val.shape)
X_seq_test_n = scaler_X.transform(
    X_seq_test.reshape(-1, N_FEATURES)).reshape(X_seq_test.shape)

scaler_y = StandardScaler()
y_train_n = scaler_y.fit_transform(y_train)
y_val_n   = scaler_y.transform(y_val)

print("\nNormalización completada.")
print(f"  X media≈0 std≈1 en train: {X_seq_train_n.mean():.4f} / {X_seq_train_n.std():.4f}")
print(f"  y media≈0 std≈1 en train: {y_train_n.mean():.4f} / {y_train_n.std():.4f}")

In [ ]:
## 2. Definición del modelo LSTM

from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam

def build_lstm(seq_len, n_features, n_targets=3):
    inp = Input(shape=(seq_len, n_features), name="seq_input")

    x = layers.LSTM(128, return_sequences=True, name="lstm_1")(inp)
    x = layers.Dropout(0.2, name="drop_1")(x)
    x = layers.LSTM(64, return_sequences=False, name="lstm_2")(x)
    x = layers.Dropout(0.2, name="drop_2")(x)
    x = layers.Dense(32, activation="relu", name="dense_hidden")(x)
    out = layers.Dense(n_targets, dtype="float32", name="output")(x)

    model = Model(inp, out, name="LSTM_trafico")
    model.compile(
        optimizer=Adam(learning_rate=3e-4),
        loss="mse",
        metrics=[
            "mae",
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
            tf.keras.metrics.R2Score(name="r2", dtype="float32"),
        ],
    )
    return model

lstm_model = build_lstm(SEQ_LEN, N_FEATURES)
lstm_model.summary()


In [ ]:
## 3. Entrenamiento

BATCH_SIZE = 4096   # revertido: batches grandes → menos actualizaciones/epoch → menos sobreajuste
EPOCHS     = 50
AUTOTUNE   = tf.data.AUTOTUNE

assert "scaler_y" in dir(), "Ejecuta la celda de normalización (§1) antes de entrenar."

# from_tensor_slices ya carga todo en RAM; .cache() es redundante y duplica memoria
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_seq_train_n, y_train_n))
    .shuffle(200_000, seed=42)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_seq_val_n, y_val_n))
    .batch(BATCH_SIZE * 2)
    .prefetch(AUTOTUNE)
)

# ── Checkpointing reanudable ─────────────────────────────────────────────────
import os, json
CKPT_BEST  = "outputs/lstm_best.keras"
CKPT_LAST  = "outputs/lstm_last.keras"
STATE_PATH = "outputs/lstm_state.json"

initial_epoch  = 0
prev_history   = None
prev_best_val  = None

if os.path.exists(CKPT_LAST) and os.path.exists(STATE_PATH):
    with open(STATE_PATH) as f:
        _state = json.load(f)
    initial_epoch = int(_state.get("last_epoch", 0))
    prev_history  = _state.get("history", {}) or {}
    if prev_history.get("val_loss"):
        prev_best_val = float(min(prev_history["val_loss"]))
    if initial_epoch >= EPOCHS:
        print(f"[checkpoint] epoch {initial_epoch} ≥ EPOCHS={EPOCHS}. "
              f"Borra {CKPT_LAST} y {STATE_PATH} para reentrenar desde cero.")
    else:
        print(f"[checkpoint] reanudando desde epoch {initial_epoch} ({CKPT_LAST})")
        lstm_model = tf.keras.models.load_model(CKPT_LAST)
else:
    print("[checkpoint] inicio desde cero (sin checkpoint previo).")

class StateSaver(tf.keras.callbacks.Callback):
    """Persiste epoch + history en JSON al final de cada epoch."""
    def __init__(self, state_path, prev_history=None):
        super().__init__()
        self.state_path = state_path
        self.history = {k: list(v) for k, v in (prev_history or {}).items()}
    def on_epoch_end(self, epoch, logs=None):
        for k, v in (logs or {}).items():
            self.history.setdefault(k, []).append(float(v))
        with open(self.state_path, "w") as f:
            json.dump({"last_epoch": epoch + 1, "history": self.history}, f)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CKPT_BEST, monitor="val_loss",
        save_best_only=True, verbose=0,
        initial_value_threshold=prev_best_val,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CKPT_LAST, save_best_only=False, verbose=0,
    ),
    StateSaver(STATE_PATH, prev_history=prev_history),
]

try:
    history = lstm_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        initial_epoch=initial_epoch,
        callbacks=callbacks,
        verbose=1,
    )
except KeyboardInterrupt:
    print(f"\n[checkpoint] interrumpido. CKPT_LAST guardado en {CKPT_LAST}. "
          f"Vuelve a ejecutar la celda para reanudar.")
    history = lstm_model.history

# Fusionar history previa (si reanudamos) para que las gráficas y métricas
# muestren la curva completa de entrenamiento, no solo desde el resume.
if prev_history:
    _keys = set(prev_history) | set(history.history)
    history.history = {
        k: list(prev_history.get(k, [])) + list(history.history.get(k, []))
        for k in _keys
    }

# ── Curvas de entrenamiento: grid 2×2 ────────────────────────────────────────
import matplotlib.pyplot as plt

if not history.history.get("loss"):
    print("No hay épocas completadas para graficar.")
else:
    n_ep      = len(history.history["loss"])
    epochs_ran = range(1, n_ep + 1)
    best_ep   = np.argmin(history.history["val_loss"]) + 1
    step      = max(1, n_ep // 10)

    metrics_cfg = [
        ("loss", "MSE Loss",  False),   # (key, label, higher_is_better)
        ("mae",  "MAE",       False),
        ("rmse", "RMSE",      False),
        ("r2",   "R² Score",  True),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()

    for ax, (metric, label, hib) in zip(axes, metrics_cfg):
        if metric not in history.history:
            ax.text(0.5, 0.5, f"{label}\nno disponible", ha="center", va="center",
                    transform=ax.transAxes, fontsize=11, color="gray")
            ax.set_title(label, fontsize=12)
            continue
        train_vals = history.history[metric]
        val_vals   = history.history[f"val_{metric}"]
        ax.plot(epochs_ran, train_vals, label="train", color="#1565C0", linewidth=2)
        ax.plot(epochs_ran, val_vals,   label="val",   color="#E65100", linewidth=2)
        ax.axvline(best_ep, color="green", linestyle="--", linewidth=1.2, alpha=0.7,
                   label=f"mejor epoch ({best_ep})")
        ax.set_title(label, fontsize=12)
        ax.set_xlabel("Epoch")
        ax.set_xticks(list(epochs_ran)[::step])
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle(
        f"LSTM — Curvas de entrenamiento  |  mejor epoch: {best_ep}  |  parada en epoch: {n_ep}",
        fontsize=13,
    )
    plt.tight_layout()
    plt.savefig("outputs/fase5_training_curve.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Mejor epoch           : {best_ep}")
    print(f"Epochs totales        : {n_ep}")
    print(f"Val MSE (mejor epoch) : {history.history['val_loss'][best_ep-1]:.5f}")
    print(f"Val MAE (mejor epoch) : {history.history['val_mae'][best_ep-1]:.5f}")
    if "val_r2" in history.history:
        print(f"Val R²  (mejor epoch) : {history.history['val_r2'][best_ep-1]:.5f}")


In [ ]:
## 4. Evaluación y comparativa Baseline / XGBoost / LSTM

from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# ── Predicciones LSTM ─────────────────────────────────────────────────────────
test_ds_pred = (
    tf.data.Dataset.from_tensor_slices(X_seq_test_n)
    .batch(BATCH_SIZE * 2).prefetch(AUTOTUNE)
)
y_pred_norm = lstm_model.predict(test_ds_pred, verbose=0)
y_pred_lstm = scaler_y.inverse_transform(y_pred_norm)   # volver a escala original

# ── Métricas LSTM ─────────────────────────────────────────────────────────────
lstm_metrics = {}
for i, target in enumerate(TARGET_COLS):
    lstm_metrics[target] = {
        "MAE":  mean_absolute_error(y_test[:, i], y_pred_lstm[:, i]),
        "RMSE": rmse(y_test[:, i], y_pred_lstm[:, i]),
    }

# ── Métricas XGBoost y Baseline (Fase 4) ─────────────────────────────────────
try:
    xgb_test = {t: xgb_metrics[("test", t)] for t in TARGET_COLS}
except NameError:
    print("xgb_metrics no encontrado — ejecuta la Fase 4 primero.")
    xgb_test = {t: {"MAE": float("nan"), "RMSE": float("nan")} for t in TARGET_COLS}

try:
    baseline_test = {t: baseline_metrics[("test", t)] for t in TARGET_COLS}
except NameError:
    baseline_test = {t: {"MAE": float("nan"), "RMSE": float("nan")} for t in TARGET_COLS}

# ── Tabla MAE: Baseline / XGB / LSTM / Δ vs XGB / Δ vs Baseline ──────────────
W = 103
print("COMPARATIVA FINAL — TEST SET (MAE)")
print("=" * W)
print(f"  {'Target':23s}  {'BASELINE':>9s}  {'XGBOOST':>9s}  {'LSTM':>9s}"
      f"  {'LSTM vs XGB':>12s}  {'LSTM vs BASE':>13s}")
print(f"  {'':23s}  {'MAE':>9s}  {'MAE':>9s}  {'MAE':>9s}"
      f"  {'Δ MAE':>12s}  {'Δ MAE':>13s}")
print("-" * W)
for target in TARGET_COLS:
    bm    = baseline_test[target]["MAE"]
    xm    = xgb_test[target]["MAE"]
    lm    = lstm_metrics[target]["MAE"]
    d_xgb  = (xm - lm) / xm  * 100 if xm  else float("nan")
    d_base = (bm - lm) / bm  * 100 if bm  else float("nan")
    print(f"  {target:23s}  {bm:9.3f}  {xm:9.3f}  {lm:9.3f}"
          f"  {d_xgb:+10.1f} %  {d_base:+11.1f} %")

# ── Tabla RMSE ────────────────────────────────────────────────────────────────
print()
print("COMPARATIVA FINAL — TEST SET (RMSE)")
print("=" * 55)
print(f"  {'Target':23s}  {'BASELINE':>9s}  {'XGBOOST':>9s}  {'LSTM':>9s}")
print(f"  {'':23s}  {'RMSE':>9s}  {'RMSE':>9s}  {'RMSE':>9s}")
print("-" * 55)
for target in TARGET_COLS:
    br = baseline_test[target]["RMSE"]
    xr = xgb_test[target]["RMSE"]
    lr = lstm_metrics[target]["RMSE"]
    print(f"  {target:23s}  {br:9.3f}  {xr:9.3f}  {lr:9.3f}")

# ── Gráfico comparativo (barras MAE) ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Comparativa MAE en Test — Baseline vs XGBoost vs LSTM", fontsize=13)

x_pos    = np.arange(3)
x_labels = ["Baseline\n(lag_1)", "XGBoost", "LSTM"]
colors   = ["#90A4AE", "#1565C0", "#B71C1C"]

for ax, target in zip(axes, TARGET_COLS):
    bm = baseline_test[target]["MAE"]
    xm = xgb_test[target]["MAE"]
    lm = lstm_metrics[target]["MAE"]
    bars = ax.bar(x_pos, [bm, xm, lm], color=colors, edgecolor="white", width=0.5)
    ax.set_title(target, fontsize=11)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=9)
    ax.set_ylabel("MAE")
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h * 1.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("outputs/fase5_comparativa_final.png", dpi=120, bbox_inches="tight")
plt.show()
print("Gráfico guardado: outputs/fase5_comparativa_final.png")
print("\nFASE 5 completada.")


In [ ]:
## 5. Guardado de métricas JSON (para comparativa final)

import json as _json
import numpy as _np
from sklearn.metrics import mean_absolute_error as _mae, mean_squared_error as _mse

def _rmse(y_true, y_pred):
    return _np.sqrt(_mse(y_true, y_pred))

_TARGET_COLS = ["intensidad_trafico", "ocupacion", "carga"]

# lstm_metrics ya calculado en celda anterior
_metrics_per_target = {}
for _i, _t in enumerate(_TARGET_COLS):
    _m = lstm_metrics[_t]["MAE"]
    _r = lstm_metrics[_t]["RMSE"]
    _metrics_per_target[_t] = {"mae": round(float(_m), 6), "rmse": round(float(_r), 6)}

_mean_mae  = float(_np.mean([v["mae"]  for v in _metrics_per_target.values()]))
_mean_rmse = float(_np.mean([v["rmse"] for v in _metrics_per_target.values()]))
_epochs_ran = len(history.history["loss"])
_stopped_early = _epochs_ran < EPOCHS

_results = {
    "model_name": "lstm_original",
    "targets": _metrics_per_target,
    "mean_mae":      round(_mean_mae,  6),
    "mean_rmse":     round(_mean_rmse, 6),
    "epochs_trained": _epochs_ran,
    "stopped_early":  _stopped_early,
}

import os as _os
_os.makedirs("outputs", exist_ok=True)
_json_path = "outputs/results_lstm_original.json"
with open(_json_path, "w") as _f:
    _json.dump(_results, _f, indent=2)

print(f"Métricas guardadas en {_json_path}")
print(_json.dumps(_results, indent=2))
